# 01 | Data quality, cleaning and SQLite

**Stakeholder and decision.** Duka Electronics' commercial lead needs an H2 inventory and marketing allocation plan. This synthetic H1 analysis compares branch, product, category, channel and monthly performance.

**Questions.** Which branches and categories contribute most? How does monthly revenue move? How do channels compare? Is the weekday/weekend basket difference distinguishable statistically?

**Success criteria.** A seeded dataset, explicit six-dimension audit, reusable cleaner with row-level action counts, and verified SQLite row counts make results reproducible. Revenue is units sold times unit price (KES); one row is one order.

This notebook audits the synthetic source, applies `src/cleaning.py`, and creates the `sales` table in `data/duka_sales.db`. Synthetic data cannot establish real-world business performance.

In [1]:
from pathlib import Path
import sys
import sqlite3
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
from cleaning import BRANCH_REGIONS, clean_sales

source_path = ROOT / "data" / "duka_sales.csv"
if not source_path.exists():
    raise FileNotFoundError(f"Run python data/generate_data.py first: {source_path}")
raw = pd.read_csv(source_path)
print(f"Raw source: {raw.shape[0]:,} rows x {raw.shape[1]} columns")
raw.head()

Raw source: 1,000 rows x 9 columns


,order_id,order_date,branch,region,product,category,channel,units_sold,unit_price
0,DUKA-0001,2026-03-24,Mombasa,Coast,Smart Watch,Wearables,Online,1,23100
1,DUKA-0002,2026-06-07,Nairobi,Capital,Laptop Pro 14,Computing,In-store,4,118900
2,DUKA-0003,2026-05-15,Nairobi,Capital,Smartphone X,Mobile,In-store,2,39700
3,DUKA-0004,2026-02-18,Thika,Central,HD Television 32in,Television,In-store,3,26600
4,DUKA-0005,2026-04-16,Kisumu,Western,Wireless Headphones,Audio,In-store,1,8400


## Six-dimension data-quality audit

Completeness checks missing cells; uniqueness checks the order key; validity checks parseability and numeric domains; consistency checks branch-region agreement and domains; accuracy is limited to plausibility checks because no independent POS source exists; timeliness checks whether the observed order dates cover the declared H1 2026 period. Accuracy cannot be independently certified for synthetic records.

In [2]:
parsed_dates = pd.to_datetime(raw["order_date"], errors="coerce")
units = pd.to_numeric(raw["units_sold"], errors="coerce")
prices = pd.to_numeric(raw["unit_price"], errors="coerce")
region_match = raw["branch"].map(BRANCH_REGIONS).eq(raw["region"])
audit = pd.DataFrame([
    {"dimension": "Completeness", "check": "Missing or blank cells", "result": int(raw.isna().sum().sum() + raw.select_dtypes(include="object").apply(lambda col: col.str.strip().eq("")).sum().sum()), "interpretation": "Counts nulls and empty strings; review every missing cell."},
    {"dimension": "Uniqueness", "check": "Duplicate order IDs", "result": int(raw["order_id"].duplicated().sum()), "interpretation": "Expected one row per order ID."},
    {"dimension": "Validity", "check": "Invalid dates or non-positive units / negative prices", "result": int((parsed_dates.isna() | units.isna() | prices.isna() | units.le(0) | prices.lt(0)).sum()), "interpretation": "Rows outside parseable/domain constraints."},
    {"dimension": "Consistency", "check": "Region agrees with branch lookup", "result": int((~region_match).sum()), "interpretation": "Branch-to-region mapping consistency exceptions."},
    {"dimension": "Accuracy", "check": "Independent source verification", "result": "Not available", "interpretation": "Synthetic values are not verifiable against a POS system; plausibility only."},
    {"dimension": "Timeliness", "check": "Observed date range vs H1 2026", "result": f"{parsed_dates.min():%Y-%m-%d} to {parsed_dates.max():%Y-%m-%d}", "interpretation": "Expected window: 2026-01-01 to 2026-06-30; dates are historical synthetic values."},
])
display(audit)

C:\Users\SEENOI HPF\AppData\Local\Temp\ipykernel_4408\2536019105.py:6: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  {"dimension": "Completeness", "check": "Missing or blank cells", "result": int(raw.isna().sum().sum() + raw.select_dtypes(include="object").apply(lambda col: col.str.strip().eq("")).sum().sum()), "interpretation": "Counts nulls and empty strings; review every missing cell."},


,dimension,check,result,interpretation
0,Completeness,Missing or blank cells,0,Counts nulls and empty strings; review every m...
1,Uniqueness,Duplicate order IDs,0,Expected one row per order ID.
2,Validity,Invalid dates or non-positive units / negative...,0,Rows outside parseable/domain constraints.
3,Consistency,Region agrees with branch lookup,0,Branch-to-region mapping consistency exceptions.
4,Accuracy,Independent source verification,Not available,Synthetic values are not verifiable against a ...
5,Timeliness,Observed date range vs H1 2026,2026-01-01 to 2026-06-30,Expected window: 2026-01-01 to 2026-06-30; dat...


## Reusable cleaning and audit log

The cleaner preserves valid orders, keeps the first duplicate key, parses dates, normalises text, derives missing regions from the documented branch mapping, validates amounts and derives revenue. Each step reports the row count before and after plus affected rows or a note.

In [3]:
clean, cleaning_log = clean_sales(raw)
display(cleaning_log)
print(f"Clean rows: {len(clean):,}; dropped: {len(raw) - len(clean):,}")
assert clean["order_id"].is_unique
assert clean["revenue"].ge(0).all()
assert clean["order_date"].notna().all()

,step,rows_before,rows_after,rows_affected,note
0,deduplicate_order_id,1000,1000,0,Kept first occurrence of each order ID.
1,parse_order_date,1000,1000,0,Removed rows with unparseable order dates.
2,derive_missing_region,1000,1000,0,Derived 0 region value(s) from the branch lookup.
3,validate_amounts,1000,1000,0,Removed missing/non-positive quantities and mi...
4,remove_missing_dimensions,1000,1000,0,Removed rows missing required descriptive fields.
5,derive_revenue,1000,1000,0,Calculated revenue as units_sold × unit_price.


Clean rows: 1,000; dropped: 0


## Load and verify SQLite clean table

In [4]:
database_path = ROOT / "data" / "duka_sales.db"
with sqlite3.connect(database_path) as connection:
    clean.to_sql("sales", connection, if_exists="replace", index=False)
    verification = pd.read_sql("SELECT COUNT(*) AS rows, COUNT(DISTINCT order_id) AS unique_orders, SUM(revenue) AS revenue_kes FROM sales", connection)
display(verification)
assert int(verification.loc[0, "rows"]) == len(clean)
print(f"SQLite table created: {database_path}")

,rows,unique_orders,revenue_kes
0,1000,1000,52464100.0


SQLite table created: C:\Users\SEENOI HPF\OneDrive\Desktop\expense tracker\postgres-performance-lab\plp-capstone-a-seenoinaurori\data\duka_sales.db
